In [1]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Install notebook dependencies in Colab runtime.
    %pip -q install rank-bm25

    from google.colab import drive
    drive.mount("/content/drive")
        

# Optional override if your repo root is non-standard.
# Example: /content/drive/MyDrive/Legal-question-answer-v1
PROJECT_ROOT = os.environ.get("PROJECT_ROOT", "")
if PROJECT_ROOT:
    print(f"Using PROJECT_ROOT from environment: {PROJECT_ROOT}")

In [2]:
import sys
from pathlib import Path

# Resolve project root in both local and Colab environments.
candidate_roots = []

# 1) Explicit env override from setup cell
project_root_env = Path(PROJECT_ROOT).expanduser() if "PROJECT_ROOT" in globals() and PROJECT_ROOT else None
if project_root_env:
    candidate_roots.append(project_root_env)

# 2) Current working directory search upward
cwd = Path.cwd().resolve()
candidate_roots.extend([cwd, *cwd.parents])

# 3) Common Colab locations
candidate_roots.extend([
    Path("/content/drive/MyDrive/Colab Notebooks/NLP/Legal-question-answer-v1"),
])

project_root = None
for path in candidate_roots:
    if (path / "utils").exists() and (path / "data").exists():
        project_root = path
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not find project root containing 'utils/' and 'data/'. "
        "Set PROJECT_ROOT env var, or update the Colab setup cell path."
    )

utils_path = str(project_root / "utils")
if utils_path not in sys.path:
    sys.path.append(utils_path)

print(f"Resolved project_root: {project_root}")

from get_jsonl_data import get_jsonl_data

Resolved project_root: E:\AI\NLP\Legal-question-answer-v1


In [3]:
chunk_store_path = project_root / "data" / "chunk_store.jsonl"
training_data_path = project_root / "data" / "training_data.jsonl"

chunk_store = get_jsonl_data(str(chunk_store_path))
training_data = get_jsonl_data(str(training_data_path))

print(chunk_store[0])
print(training_data[0])

{'doc_id': 'doc_0', 'article_id': 'article_0', 'chunk_id': 0, 'text': 'Điều 5. Nguyên tắc hội nhập và hợp tác quốc tế về địa chất, khoáng sản'}
{'doc_id': 'doc_0', 'article_id': 'article_0', 'question': 'Theo Điều 5 Luật Địa chất và Khoáng sản, hội nhập và hợp tác quốc tế về địa chất, khoáng sản được thực hiện trong những hoạt động cụ thể nào?', 'answer': 'Theo Khoản 1 Điều 5 Luật Địa chất và Khoáng sản, hội nhập và hợp tác quốc tế về địa chất, khoáng sản được thực hiện trong các hoạt động sau: nghiên cứu, điều tra cơ bản địa chất; điều tra địa chất về khoáng sản; hoạt động khoáng sản; và quản lý hoạt động khoáng sản.', 'positive_chunks': ['1. Hội nhập và hợp tác quốc tế trong nghiên cứu, điều tra cơ bản địa chất, điều tra địa chất về khoáng sản, hoạt động khoáng sản, quản lý hoạt động khoáng sản phải đặt trong tổng thể chiến lược phát triển kinh tế - xã hội của đất nước trong từng thời kỳ; chiến lược địa chất, khoáng sản và công nghiệp khai khoáng;', 'Điều 5. Nguyên tắc hội nhập và hợ

In [7]:
from collections import defaultdict
from rank_bm25 import BM25Okapi
import numpy as np


def normalize_whitespace(text):
    return " ".join((text or "").split())


top_k = 10
top_k_neg = 5
progress_every_rows = 100
total_rows = len(training_data)

# Pre-index chunks by doc_id with normalized text and tokens.
chunks_by_doc = defaultdict(list)
for chunk in chunk_store:
    doc_id = chunk.get("doc_id")
    article_id = chunk.get("article_id")
    chunk_text = normalize_whitespace(chunk.get("text"))
    if doc_id is not None and chunk_text:
        chunks_by_doc[doc_id].append({
            "article_id": article_id,
            "text": chunk_text,
            "tokens": chunk_text.split(),
        })

print(f"Starting negative chunk generation for {total_rows} training rows...")

empty_negatives_count = 0
question_tokens_cache = {}
insufficient_negatives  = []

for row_idx, row in enumerate(training_data, start=1):
    doc_id = row.get("doc_id")
    article_id = row.get("article_id")
    question = normalize_whitespace(row.get("question"))
    positive_chunks = row.get("positive_chunks", [])

    if not question or not positive_chunks:
        row["negative_chunks"] = []
        empty_negatives_count += 1
    else:
        # Memory-stable path: do not cache per (doc_id, article_id) candidate corpora.
        # Caching those corpora causes near-duplicate large lists and can exhaust RAM.
        doc_chunks = chunks_by_doc.get(doc_id, [])
        base_texts = [
            ch["text"]
            for ch in doc_chunks
            if ch.get("article_id") != article_id
        ]

        base_tokens = [
            ch["tokens"]
            for ch in doc_chunks
            if ch.get("article_id") != article_id
        ]

        positive_norm = [normalize_whitespace(t) for t in positive_chunks if t]
        positive_norm = [t for t in positive_norm if t]
        positive_set = set(positive_norm)

        # Keep positives in BM25 corpus as requested.
        corpus_texts = base_texts + positive_norm
        corpus_texts_set = set(corpus_texts)
        corpus_tokens = base_tokens + [t.split() for t in positive_norm]
        corpus_tokens_set = set(tuple(tokens) for tokens in corpus_tokens)

        if not corpus_tokens_set:
            row["negative_chunks"] = []
            empty_negatives_count += 1
        else:
            if question in question_tokens_cache:
                tokenized_question = question_tokens_cache[question]
            else:
                tokenized_question = question.split()
                question_tokens_cache[question] = tokenized_question

            bm25 = BM25Okapi(corpus_tokens_set)
            bm25_scores = bm25.get_scores(tokenized_question)

            k = min(top_k, len(bm25_scores))
            if k == 0:
                row["negative_chunks"] = []
                empty_negatives_count += 1
            else:
                if k == len(bm25_scores):
                    top_k_idx = np.argsort(bm25_scores)[::-1]
                else:
                    top_k_idx = np.argpartition(bm25_scores, -k)[-k:]
                    top_k_idx = top_k_idx[np.argsort(bm25_scores[top_k_idx])[::-1]]

                top_k_chunks = [list(corpus_texts_set)[idx] for idx in top_k_idx]
                negative_chunks = [t for t in top_k_chunks if t not in positive_set][:top_k_neg]
                if not negative_chunks:
                    empty_negatives_count += 1
                if len(negative_chunks) < top_k_neg:
                    # print(f"Warning: Only found {len(negative_chunks)} negative chunks for question '{question}'.")
                    # print(f"positive_chunks: {positive_chunks}.")
                    # print(f"top_k_chunks: {top_k_chunks}.")
                    insufficient_negatives.append((doc_id, article_id, question, len(negative_chunks)))

                    # Add more negatives from other documents if we have fewer than top_k_neg.
                    candidate_doc_ids = [
                        d for d in chunks_by_doc.keys()
                        if d != doc_id and chunks_by_doc.get(d)
                    ]
                    seen_negative_set = set(negative_chunks)
                    attempts = 0
                    max_attempts = max(1, len(candidate_doc_ids) * 2)

                    while len(negative_chunks) < top_k_neg and candidate_doc_ids and attempts < max_attempts:
                        random_doc_id = np.random.choice(candidate_doc_ids)
                        random_doc_chunks = chunks_by_doc.get(random_doc_id, [])
                        needed_negatives = top_k_neg - len(negative_chunks)

                        additional_chunks = []
                        for chunk in random_doc_chunks:
                            chunk_text = chunk["text"]
                            if chunk_text and chunk_text not in positive_set and chunk_text not in seen_negative_set:
                                additional_chunks.append(chunk_text)
                                seen_negative_set.add(chunk_text)
                            if len(additional_chunks) >= needed_negatives:
                                break

                        negative_chunks.extend(additional_chunks)
                        attempts += 1

                row["negative_chunks"] = negative_chunks


    if row_idx % progress_every_rows == 0 or row_idx == total_rows:
        print(
            f"[Progress] rows={row_idx}/{total_rows} | "
            f"docs_seen={len(chunks_by_doc)} | "
            f"q_cache={len(question_tokens_cache)} | "
            f"empty_negatives={empty_negatives_count}"
        )

print(
    f"Done generating negative chunks for {total_rows} rows. "
    f"Empty negatives: {empty_negatives_count}."
    )

Starting negative chunk generation for 29145 training rows...
[Progress] rows=100/29145 | docs_seen=328 | q_cache=100 | empty_negatives=0
[Progress] rows=200/29145 | docs_seen=328 | q_cache=200 | empty_negatives=0
[Progress] rows=300/29145 | docs_seen=328 | q_cache=300 | empty_negatives=0
[Progress] rows=400/29145 | docs_seen=328 | q_cache=400 | empty_negatives=3
[Progress] rows=500/29145 | docs_seen=328 | q_cache=500 | empty_negatives=3
[Progress] rows=600/29145 | docs_seen=328 | q_cache=600 | empty_negatives=3
[Progress] rows=700/29145 | docs_seen=328 | q_cache=700 | empty_negatives=6
[Progress] rows=800/29145 | docs_seen=328 | q_cache=800 | empty_negatives=6
[Progress] rows=900/29145 | docs_seen=328 | q_cache=900 | empty_negatives=6
[Progress] rows=1000/29145 | docs_seen=328 | q_cache=1000 | empty_negatives=6
[Progress] rows=1100/29145 | docs_seen=328 | q_cache=1100 | empty_negatives=6
[Progress] rows=1200/29145 | docs_seen=328 | q_cache=1200 | empty_negatives=12
[Progress] rows=130

In [10]:
print(f"Length of insufficient_negatives: {len(insufficient_negatives)}")
insufficient_negatives[:5]

Length of insufficient_negatives: 543


[('doc_2',
  'article_114',
  'Theo Khoản 1 Điều 111 của Luật Địa chất và Khoáng sản số 54/2024/QH15, tổ chức, cá nhân đã nộp hồ sơ hành chính về địa chất, khoáng sản trước ngày Luật này có hiệu lực mà chưa được giải quyết sẽ có những lựa chọn nào?',
  0),
 ('doc_2',
  'article_114',
  'Hãy phân tích các quy định chuyển tiếp của Luật Địa chất và Khoáng sản số 54/2024/QH15 liên quan đến việc xử lý các giấy phép, quyết định trong hoạt động khoáng sản đã được cấp trước ngày Luật này có hiệu lực thi hành, bao gồm cả trường hợp gia hạn, điều chỉnh và các giấy phép không phù hợp với quy định mới.',
  0),
 ('doc_2',
  'article_114',
  'Một công ty khai thác khoáng sản được cấp giấy phép khai thác trước ngày 01 tháng 7 năm 2025. Giấy phép này có một số nội dung về công suất khai thác không phù hợp với quy định tại Khoản 2 Điều 56 của Luật Địa chất và Khoáng sản số 54/2024/QH15. Công ty này cần phải làm gì để tuân thủ quy định mới và điều gì sẽ xảy ra nếu công ty không thực hiện đúng thời hạn?'

In [11]:
# Save the training data to a JSONL file
import json

output_path = project_root / "data" / "training_data.jsonl"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    for sample in training_data:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print(f"Saved to: {output_path}")

Saved to: E:\AI\NLP\Legal-question-answer-v1\data\training_data.jsonl
